Для emotions

1.1 Задание

In [90]:
import nltk
import string
from datasets import load_dataset
from nltk.corpus import wordnet, stopwords
import pandas as pd
from sklearn.datasets import fetch_20newsgroups
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier
from sklearn.metrics import f1_score, classification_report, confusion_matrix
from sklearn.pipeline import Pipeline

In [4]:
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('wordnet')
nltk.download('averaged_perceptron_tagger_eng')
nltk.download('omw-1.4')
nltk.download('stopwords')
stop_words = set(stopwords.words('english'))

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /root/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger_eng.zip.
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


In [5]:
dataset = load_dataset('emotion')
ex = dataset['train'][0]
txt = ex['text']
txt

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

split/train-00000-of-00001.parquet:   0%|          | 0.00/1.03M [00:00<?, ?B/s]

split/validation-00000-of-00001.parquet:   0%|          | 0.00/127k [00:00<?, ?B/s]

split/test-00000-of-00001.parquet:   0%|          | 0.00/129k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/16000 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/2000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/2000 [00:00<?, ? examples/s]

'i didnt feel humiliated'

In [6]:
tokens = nltk.word_tokenize(txt)
tokens

['i', 'didnt', 'feel', 'humiliated']

In [7]:
def get_wordnet_pos(tag):
  if tag.startswith('J'):
    return wordnet.ADJ
  elif tag.startswith('V'):
    return wordnet.VERB
  elif tag.startswith('R'):
    return wordnet.ADV
  else:
    return wordnet.NOUN

In [8]:
def preprocess_with_stemming(text):
  text = text.lower()
  text = text.translate(str.maketrans('','',string.punctuation))
  tokens = nltk.word_tokenize(text)
  stemmer = nltk.PorterStemmer()
  stemmer_tokens = [stemmer.stem(token) for token in tokens]
  stemmer_tokens = [token for token in stemmer_tokens if token not in stop_words]
  return stemmer_tokens

In [9]:
def preprocess_with_lemmatization(text):
  text = text.lower()
  text = text.translate(str.maketrans('','',string.punctuation))
  tokens = nltk.word_tokenize(text)
  tagged = nltk.pos_tag(tokens)
  lemmatizer = nltk.WordNetLemmatizer()
  lemmatized_tokens = [lemmatizer.lemmatize(token, get_wordnet_pos(tag)) for token,tag in tagged if token not in string.punctuation]
  lemmatized_tokens = [token for token in lemmatized_tokens if token not in stop_words]
  return lemmatized_tokens

In [10]:
stemmed_result = preprocess_with_stemming(txt)
lemmatized_result = preprocess_with_lemmatization(txt)

In [11]:
stemmed_result

['didnt', 'feel', 'humili']

In [12]:
lemmatized_result

['didnt', 'feel', 'humiliate']

Вариант 1

In [13]:
from sklearn.feature_extraction.text import CountVectorizer

In [14]:
reviews = [
    "I loved this movie, it was amazing and wonderful",
    "The movie was boring and terrible, I hated it",
    "Amazing acting and great story, loved the characters",
    "Terrible movie, boring from start to finish, hated every minute",
    "Great film with wonderful performances and amazing visuals"
]

lemmatizer = nltk.WordNetLemmatizer()
lemmatized_result = []

for review in reviews:
    review_no_punct = review.lower().translate(str.maketrans('','',string.punctuation))
    words = nltk.word_tokenize(review_no_punct)
    tagged = nltk.pos_tag(words)

    lemmatized_words = []
    for word, tag in tagged:
        if word not in string.punctuation:
            wordnet_pos = get_wordnet_pos(tag)
            lemma = lemmatizer.lemmatize(word, wordnet_pos)
            if lemma not in stop_words:
                lemmatized_words.append(lemma)

    lemmatized_result.append(' '.join(lemmatized_words))

vectorizer = CountVectorizer(binary=True)
X = vectorizer.fit_transform(lemmatized_result)

In [15]:
X.toarray()

array([[0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 1],
       [0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 1, 0, 0],
       [1, 1, 0, 0, 1, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0],
       [0, 0, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 1, 0, 1, 0, 1, 0, 0],
       [0, 0, 1, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 1, 1]])

Вариант 2

In [16]:
vectorizer = CountVectorizer(binary=True, tokenizer=preprocess_with_lemmatization, lowercase=False)
X = vectorizer.fit_transform(reviews)

/usr/local/lib/python3.12/dist-packages/sklearn/feature_extraction/text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


In [17]:
X.toarray()

array([[0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 1],
       [0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 1, 0, 0],
       [1, 1, 0, 0, 1, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0],
       [0, 0, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 1, 0, 1, 0, 1, 0, 0],
       [0, 0, 1, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 1, 1]])

Вариант 3

In [18]:
vectorizer = CountVectorizer(binary=True)
X = vectorizer.fit_transform(reviews)

In [19]:
X.toarray()

array([[0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 1, 0, 0, 0, 0, 0, 1, 0,
        0, 1, 0, 1],
       [0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 1, 0, 0, 0, 1, 1, 0, 0,
        0, 1, 0, 0],
       [1, 1, 1, 0, 1, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 1, 0, 1, 0, 0,
        0, 0, 0, 0],
       [0, 0, 0, 1, 0, 1, 0, 1, 1, 0, 1, 0, 0, 1, 1, 0, 1, 0, 1, 0, 0, 1,
        0, 0, 0, 0],
       [0, 1, 1, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0,
        1, 0, 1, 1]])

In [20]:
for i in range(3):
  print(f"{i+11})")
  ex = dataset['train'][i+11]
  txt = ex['text']
  print(f"text -{txt}")
  tokens = nltk.word_tokenize(txt)
  print(f"tokens - {tokens}")
  stemmed_result = preprocess_with_stemming(txt)
  lemmatized_result = preprocess_with_lemmatization(txt)
  print(f"stemmed - {stemmed_result}\n lemmed - {lemmatized_result}")



11)
text -i do feel that running is a divine experience and that i can expect to have some type of spiritual encounter
tokens - ['i', 'do', 'feel', 'that', 'running', 'is', 'a', 'divine', 'experience', 'and', 'that', 'i', 'can', 'expect', 'to', 'have', 'some', 'type', 'of', 'spiritual', 'encounter']
stemmed - ['feel', 'run', 'divin', 'experi', 'expect', 'type', 'spiritu', 'encount']
 lemmed - ['feel', 'run', 'divine', 'experience', 'expect', 'type', 'spiritual', 'encounter']
12)
text -i think it s the easiest time of year to feel dissatisfied
tokens - ['i', 'think', 'it', 's', 'the', 'easiest', 'time', 'of', 'year', 'to', 'feel', 'dissatisfied']
stemmed - ['think', 'easiest', 'time', 'year', 'feel', 'dissatisfi']
 lemmed - ['think', 'easy', 'time', 'year', 'feel', 'dissatisfied']
13)
text -i feel low energy i m just thirsty
tokens - ['i', 'feel', 'low', 'energy', 'i', 'm', 'just', 'thirsty']
stemmed - ['feel', 'low', 'energi', 'thirsti']
 lemmed - ['feel', 'low', 'energy', 'thirsty']


1.2

In [21]:
print('stemmed lemmed')
for i in range(3):
  ex = dataset['train'][i+1]
  txt = ex['text']
  tokens = nltk.word_tokenize(txt)
  stemmed_result = preprocess_with_stemming(txt)
  lemmatized_result = preprocess_with_lemmatization(txt)
  for j in range(len(stemmed_result)):
      print(f"{j+1}) - {stemmed_result[j]} - {lemmatized_result[j]}")

stemmed lemmed
1) - go - go
2) - feel - feel
3) - hopeless - hopeless
4) - damn - damned
5) - hope - hopeful
6) - around - around
7) - someon - someone
8) - care - care
9) - awak - awake
1) - im - im
2) - grab - grab
3) - minut - minute
4) - post - post
5) - feel - feel
6) - greedi - greedy
7) - wrong - wrong
1) - ever - ever
2) - feel - feel
3) - nostalg - nostalgic
4) - fireplac - fireplace
5) - know - know
6) - still - still
7) - properti - property


Стемминг грубо отрезает концы слов, а лемматизация более осмысленно меняет форму слова

1.3

In [22]:
text = "The weather is good, let's go to the park?"
text1 = text.translate(str.maketrans('','',string.punctuation))
tokens = nltk.word_tokenize(text1)
print(tokens)
tokens = nltk.word_tokenize(text)
tokens = [t for t in tokens if t not in string.punctuation]
print(tokens)


['The', 'weather', 'is', 'good', 'lets', 'go', 'to', 'the', 'park']
['The', 'weather', 'is', 'good', 'let', "'s", 'go', 'to', 'the', 'park']


Слова с ', - , числа с плавающей точкой и тп могут быть соединены в одном из подходов

Для Newsgroups

In [23]:
from sklearn.datasets import fetch_20newsgroups

In [24]:
cat = ['comp.sys.ibm.pc.hardware',
'comp.sys.mac.hardware',
'comp.graphics',
'comp.windows.x']

In [25]:
data = fetch_20newsgroups(
    subset='train',
    categories=cat
)

In [26]:
txt = data.data[0]
txt

'From: bgrubb@dante.nmsu.edu (GRUBB)\nSubject: Re: IDE vs SCSI\nOrganization: New Mexico State University, Las Cruces, NM\nLines: 49\nDistribution: world\nNNTP-Posting-Host: dante.nmsu.edu\n\nDXB132@psuvm.psu.edu writes:\n>SCSI-I ranges from 0-5MB/s.\n>SCSI-II ranges from 0-40MB/s.\n>IDE ranges from 0-8.3MB/s.                                       \n>ESDI is always 1.25MB/s (although there are some non-standard versions)\nThe above does not tell the proper story of SCSI:\nSCSI-I: 8-bit asynchronous {~1.5MB/s ave}, synchronous {5MB/s max} transfer \nbase.\nSCSI-1{faster} this requires a SCSI-2 controller chip and provides\n SCSI-2 {8-bit to 16-bit} speeds with SCSI-1 controlers.\nSCSI-2: 4-6MB/s with 10MB/s burst{8-bit}, 8-12MB/s with 20MB/s burst {16-bit}, \nand 15-20MB/s with 40MB/s burst{32-bit/wide and fast}.  16-bit SCSI can be\nwide or fast, it depends on how the port is designed{The Quadras will support\nfast SCSI but not wide when the OS SCSI manager is rewritten since the\nQuar

In [27]:
tokens = nltk.word_tokenize(txt)
tokens

['From',
 ':',
 'bgrubb',
 '@',
 'dante.nmsu.edu',
 '(',
 'GRUBB',
 ')',
 'Subject',
 ':',
 'Re',
 ':',
 'IDE',
 'vs',
 'SCSI',
 'Organization',
 ':',
 'New',
 'Mexico',
 'State',
 'University',
 ',',
 'Las',
 'Cruces',
 ',',
 'NM',
 'Lines',
 ':',
 '49',
 'Distribution',
 ':',
 'world',
 'NNTP-Posting-Host',
 ':',
 'dante.nmsu.edu',
 'DXB132',
 '@',
 'psuvm.psu.edu',
 'writes',
 ':',
 '>',
 'SCSI-I',
 'ranges',
 'from',
 '0-5MB/s',
 '.',
 '>',
 'SCSI-II',
 'ranges',
 'from',
 '0-40MB/s',
 '.',
 '>',
 'IDE',
 'ranges',
 'from',
 '0-8.3MB/s',
 '.',
 '>',
 'ESDI',
 'is',
 'always',
 '1.25MB/s',
 '(',
 'although',
 'there',
 'are',
 'some',
 'non-standard',
 'versions',
 ')',
 'The',
 'above',
 'does',
 'not',
 'tell',
 'the',
 'proper',
 'story',
 'of',
 'SCSI',
 ':',
 'SCSI-I',
 ':',
 '8-bit',
 'asynchronous',
 '{',
 '~1.5MB/s',
 'ave',
 '}',
 ',',
 'synchronous',
 '{',
 '5MB/s',
 'max',
 '}',
 'transfer',
 'base',
 '.',
 'SCSI-1',
 '{',
 'faster',
 '}',
 'this',
 'requires',
 'a',
 'SC

In [28]:
stemmed_result = preprocess_with_stemming(txt)
lemmatized_result = preprocess_with_lemmatization(txt)

In [29]:
stemmed_result

['bgrubbdantenmsuedu',
 'grubb',
 'subject',
 'ide',
 'vs',
 'scsi',
 'organ',
 'new',
 'mexico',
 'state',
 'univers',
 'la',
 'cruce',
 'nm',
 'line',
 '49',
 'distribut',
 'world',
 'nntppostinghost',
 'dantenmsuedu',
 'dxb132psuvmpsuedu',
 'write',
 'scsii',
 'rang',
 '05mb',
 'scsiii',
 'rang',
 '040mb',
 'ide',
 'rang',
 '083mb',
 'esdi',
 'alway',
 '125mb',
 'although',
 'nonstandard',
 'version',
 'abov',
 'doe',
 'tell',
 'proper',
 'stori',
 'scsi',
 'scsii',
 '8bit',
 'asynchron',
 '15mb',
 'ave',
 'synchron',
 '5mb',
 'max',
 'transfer',
 'base',
 'scsi1fast',
 'thi',
 'requir',
 'scsi2',
 'control',
 'chip',
 'provid',
 'scsi2',
 '8bit',
 '16bit',
 'speed',
 'scsi1',
 'control',
 'scsi2',
 '46mb',
 '10mb',
 'burst8bit',
 '812mb',
 '20mb',
 'burst',
 '16bit',
 '1520mb',
 '40mb',
 'burst32bitwid',
 'fast',
 '16bit',
 'scsi',
 'wide',
 'fast',
 'depend',
 'port',
 'designedth',
 'quadra',
 'support',
 'fast',
 'scsi',
 'wide',
 'os',
 'scsi',
 'manag',
 'rewritten',
 'sinc',


In [30]:
lemmatized_result

['bgrubbdantenmsuedu',
 'grubb',
 'subject',
 'ide',
 'v',
 'scsi',
 'organization',
 'new',
 'mexico',
 'state',
 'university',
 'la',
 'crux',
 'nm',
 'line',
 '49',
 'distribution',
 'world',
 'nntppostinghost',
 'dantenmsuedu',
 'dxb132psuvmpsuedu',
 'write',
 'scsii',
 'range',
 '05mbs',
 'scsiii',
 'range',
 '040mbs',
 'ide',
 'range',
 '083mbs',
 'esdi',
 'always',
 '125mbs',
 'although',
 'nonstandard',
 'version',
 'tell',
 'proper',
 'story',
 'scsi',
 'scsii',
 '8bit',
 'asynchronous',
 '15mbs',
 'ave',
 'synchronous',
 '5mbs',
 'max',
 'transfer',
 'base',
 'scsi1faster',
 'require',
 'scsi2',
 'controller',
 'chip',
 'provide',
 'scsi2',
 '8bit',
 '16bit',
 'speed',
 'scsi1',
 'controlers',
 'scsi2',
 '46mbs',
 '10mbs',
 'burst8bit',
 '812mbs',
 '20mbs',
 'burst',
 '16bit',
 '1520mbs',
 '40mbs',
 'burst32bitwide',
 'fast',
 '16bit',
 'scsi',
 'wide',
 'fast',
 'depend',
 'port',
 'designedthe',
 'quadras',
 'support',
 'fast',
 'scsi',
 'wide',
 'scsi',
 'manager',
 'rewri

In [31]:
for i in range(3):
  print(f"{i+11})")
  txt = data.data[i+11]
  tokens = nltk.word_tokenize(txt)
  print(f"tokens - {tokens}")
  stemmed_result = preprocess_with_stemming(txt)
  lemmatized_result = preprocess_with_lemmatization(txt)
  print(f"stemmed - {stemmed_result}\n lemmed - {lemmatized_result}")

11)
tokens - ['From', ':', 'Robert', 'Everett', 'Brunskill', '<', 'rb6t+', '@', 'andrew.cmu.edu', '>', 'Subject', ':', 'Re', ':', '$', '$', '$', 'to', 'fix', 'TRACKBALL', 'Organization', ':', 'Freshman', ',', 'Electrical', 'and', 'Computer', 'Engineering', ',', 'Carnegie', 'Mellon', ',', 'Pittsburgh', ',', 'PA', 'Lines', ':', '5', 'NNTP-Posting-Host', ':', 'po2.andrew.cmu.edu', 'In-Reply-To', ':', '<', '93105.152944BR4416A', '@', 'auvm.american.edu', '>', 'The', 'little', 'blue', 'roller', 'on', 'the', 'trackball', 'interior', 'is', 'probably', 'rubbing', 'against', 'its', 'support', ',', 'just', 'push', 'it', 'down', 'the', 'pin', 'so', 'that', 'it', 'no', 'longer', 'touches', 'it', '.', 'I', 'had', 'a', 'similar', 'problem', '.', 'Rob']
stemmed - ['robert', 'everett', 'brunskil', 'rb6tandrewcmuedu', 'subject', 'fix', 'trackbal', 'organ', 'freshman', 'electr', 'comput', 'engin', 'carnegi', 'mellon', 'pittsburgh', 'pa', 'line', '5', 'nntppostinghost', 'po2andrewcmuedu', 'inreplyto', '9

1.2

In [32]:
print('stemmed lemmed')
for i in range(2):
  txt = data.data[i+1]
  tokens = nltk.word_tokenize(txt)
  stemmed_result = preprocess_with_stemming(txt)
  lemmatized_result = preprocess_with_lemmatization(txt)
  for j in range(len(stemmed_result) - 5):
      print(f"{j+1}) - {stemmed_result[j]} - {lemmatized_result[j]}")

stemmed lemmed
1) - donhcupportalcom - donhcupportalcom
2) - hirschfeld - hirschfeld
3) - subject - subject
4) - toshiba - toshiba
5) - 3401b - 3401b
6) - cdrom - cdrom
7) - ani - problem
8) - problem - organization
9) - organ - portal
10) - portal - system
11) - system - tm
12) - tm - line
13) - line - 1
14) - 1 - pas16
1) - christycsconcordiaca - christycsconcordiaca
2) - christi - christy
3) - subject - subject
4) - xfree86 - xfree86
5) - need - need
6) - help - help
7) - organ - organization
8) - comput - computer
9) - scienc - science
10) - concordia - concordia
11) - univers - university
12) - montreal - montreal
13) - quebec - quebec
14) - line - line
15) - 23 - 23
16) - hi - hi
17) - got - get
18) - xfree86 - xfree86
19) - run - run
20) - pc - pc
21) - consensi - consensys
22) - encount - encounter
23) - minor - minor
24) - hope - hope
25) - probem - probems
26) - pc - pc
27) - hook - hook
28) - lan - lan
29) - want - want
30) - remot - remote
31) - x - x
32) - applic - applica

3 Задание

Задание следующее - посмотреть внимательно какие слова в каком датасете представляют текст и подумать, есть ли среди них сепец. символы, спец.слова разметки, которые повторяются из текста в текст и от которых потенциально текст нужно очищать. И почитить текст.

Первый датасет

In [33]:
!pip install emoji

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 608.4/608.4 kB 26.7 MB/s eta 0:00:00


In [34]:
import re
import emoji
from collections import Counter

In [75]:
dataset = load_dataset('emotion')
df = pd.DataFrame(dataset['train'])

In [76]:
for i in range(100):
  print(f"{df['text'].iloc[i][:100]} - {df['label'].iloc[i]}")


i didnt feel humiliated - 0
i can go from feeling so hopeless to so damned hopeful just from being around someone who cares and  - 0
im grabbing a minute to post i feel greedy wrong - 3
i am ever feeling nostalgic about the fireplace i will know that it is still on the property - 2
i am feeling grouchy - 3
ive been feeling a little burdened lately wasnt sure why that was - 0
ive been taking or milligrams or times recommended amount and ive fallen asleep a lot faster but i a - 5
i feel as confused about life as a teenager or as jaded as a year old man - 4
i have been with petronas for years i feel that petronas has performed well and made a huge profit - 1
i feel romantic too - 2
i feel like i have to make the suffering i m seeing mean something - 0
i do feel that running is a divine experience and that i can expect to have some type of spiritual e - 1
i think it s the easiest time of year to feel dissatisfied - 3
i feel low energy i m just thirsty - 0
i have immense sympathy with the g

In [77]:
def find_repeat(texts, min_length=3, max_length=20, top_n=30):
    all_text = ' '.join([str(t) for t in texts if isinstance(t, str)])
    words = all_text.split()
    word_freq = Counter(words)
    return word_freq

word_freq = find_repeat(df['text'].tolist())

for word, count in word_freq.most_common(30):
    print(f"  '{word}': {count}")

  'i': 25859
  'feel': 11183
  'and': 9589
  'to': 8972
  'the': 8370
  'a': 6200
  'feeling': 5112
  'that': 5112
  'of': 4990
  'my': 4283
  'in': 3433
  'it': 3127
  'like': 2908
  'so': 2527
  'for': 2431
  'im': 2430
  'me': 2309
  'but': 2255
  'was': 2227
  'have': 2224
  'is': 2184
  'this': 2088
  'am': 2082
  'with': 2015
  'not': 1827
  'about': 1795
  'be': 1778
  'as': 1565
  'on': 1551
  'you': 1471


In [78]:
import nltk
from nltk.corpus import stopwords
nltk.download('stopwords', quiet=True)
nltk.download('punkt', quiet=True)

def clean(text):
    original = text
    cleaned = text
    cleaned = cleaned.lower()
    cleaned = re.sub(r'@\w+', ' ', cleaned)
    cleaned = re.sub(r'#(\w+)', r'\1', cleaned)
    cleaned = re.sub(r':[a-z_]+:', ' ', cleaned)
    cleaned = re.sub(r'[^\w\s]', ' ', cleaned)
    cleaned = re.sub(r'\s+', ' ', cleaned)
    cleaned = cleaned.strip()
    words = cleaned.split()
    stop_words = set(stopwords.words('english'))
    words = [w for w in words if w not in stop_words]
    cleaned = ' '.join(words)

    return cleaned

test_texts = df['text'].iloc[:5].tolist()
for i, text in enumerate(test_texts):
    print(f"\nОригинал {i+1}: {text[:100]}")
    cleaned = clean(text)
    print(f"Очищенный {i+1}: {cleaned[:100]}")


Оригинал 1: i didnt feel humiliated
Очищенный 1: didnt feel humiliated

Оригинал 2: i can go from feeling so hopeless to so damned hopeful just from being around someone who cares and 
Очищенный 2: go feeling hopeless damned hopeful around someone cares awake

Оригинал 3: im grabbing a minute to post i feel greedy wrong
Очищенный 3: im grabbing minute post feel greedy wrong

Оригинал 4: i am ever feeling nostalgic about the fireplace i will know that it is still on the property
Очищенный 4: ever feeling nostalgic fireplace know still property

Оригинал 5: i am feeling grouchy
Очищенный 5: feeling grouchy


In [79]:
cleaned_df = df.copy()
cleaned_df['cleaned_text'] = cleaned_df['text'].apply(
        lambda x: clean(x)
    )

4

In [82]:
def filter_by_pos(text, keep_pos_tags):
    text_clean = text.lower()
    tokens = nltk.word_tokenize(text_clean)
    words_only = [token for token in tokens if token.isalpha()]
    tagged_tokens = nltk.pos_tag(words_only)
    filtered_words = [
        word for word, tag in tagged_tokens
        if any(tag.startswith(prefix) for prefix in keep_pos_tags)
    ]
    return ' '.join(filtered_words)

test_text = df['text'].iloc[1][:500]
print(f"Исходный текст:\n{test_text}\n")
nouns_adjs = filter_by_pos(test_text, ['N', 'J'])
nouns_adjs_verbs = filter_by_pos(test_text, ['N', 'J', 'V'])
print(nouns_adjs)
print(nouns_adjs_verbs)

Исходный текст:
i can go from feeling so hopeless to so damned hopeful just from being around someone who cares and is awake

i hopeless damned hopeful someone awake
i go feeling hopeless damned hopeful being someone cares is awake


In [83]:
df['nouns_adjs'] = df['text'].apply(lambda x: filter_by_pos(x, ['N', 'J']))
df['nouns_adjs_verbs'] = df['text'].apply(lambda x: filter_by_pos(x, ['N', 'J', 'V']))
for i in range(3):
    print(f"\nДокумент {i+1}")
    print(f"Оригинал: {df['text'].iloc[i][:200]}")
    print(f"Сущ+прил: {df['nouns_adjs'].iloc[i][:200]}")
    print(f"Сущ+прил+гл: {df['nouns_adjs_verbs'].iloc[i][:200]}")


Документ 1
Оригинал: i didnt feel humiliated
Сущ+прил: i feel
Сущ+прил+гл: i didnt feel humiliated

Документ 2
Оригинал: i can go from feeling so hopeless to so damned hopeful just from being around someone who cares and is awake
Сущ+прил: i hopeless damned hopeful someone awake
Сущ+прил+гл: i go feeling hopeless damned hopeful being someone cares is awake

Документ 3
Оригинал: im grabbing a minute to post i feel greedy wrong
Сущ+прил: im minute i feel greedy wrong
Сущ+прил+гл: im grabbing minute post i feel greedy wrong


Второй датасет

In [84]:
df = pd.DataFrame({
    'text': data.data,
    'category': [data.target_names[i] for i in data.target],
    'label': data.target
})

In [85]:
for i in range(100):
  print(f"{df['text'].iloc[i][:100]} - {df['label'].iloc[i]}")

From: bgrubb@dante.nmsu.edu (GRUBB)
Subject: Re: IDE vs SCSI
Organization: New Mexico State Universi - 1
From: DonH@cup.portal.com (Don - Hirschfeld)
Subject: Re: Toshiba 3401B CD-ROM:  Any problems?
Organ - 1
From: christy@cs.concordia.ca (Christy)
Subject: XFree86 --- need help...
Organization: Computer Sci - 3
From: arp@cooper!osd (Andrew Pinkowitz)
Subject: SIGGRAPH -- Conference on Understanding Images
Keyw - 0
From: Peter.vanderveen@visser.el.wau.nl  (Peter van der Veen)
Subject: Re: Fonts in POV??
Lines: 30
 - 0
From: pmcgilla@hp.uwsuper.edu (Mr. Patrick L. McGillan)
Subject: DXF format display
Organization: Un - 3
From: afung@athena.mit.edu (Archon Fung)
Subject: wrong RAM in Duo?
Organization: Massachusetts Inst - 2
From: kiran@village.com (Kiran Wagle)
Subject: Replacing internal FDHD w/ floptical?
Organization: t - 2
From: d88-jwa@hemul.nada.kth.se (Jon Wtte)
Subject: Re: Please Recommend 3D Graphics Library For Mac - 0
From: sgoldste@aludra.usc.edu (Fogbound Child)
Subject:

In [86]:
def analyze(texts, sample_size=500):
    sample_texts = texts[:sample_size] if len(texts) > sample_size else texts
    patterns = {
        'email': re.compile(r'[\w\.-]+@[\w\.-]+\.\w+'),
        'url': re.compile(r'http[s]?://(?:[a-zA-Z]|[0-9]|[$-_@.&+]|[!*\\(\\),]|(?:%[0-9a-fA-F][0-9a-fA-F]))+'),
        'ip_address': re.compile(r'\b(?:[0-9]{1,3}\.){3}[0-9]{1,3}\b'),
        'path': re.compile(r'(?:[a-zA-Z]:)?(?:[\\/][\w\s.-]+)+[\\/]?'),
        'quote_marker': re.compile(r'^>+.*$', re.MULTILINE),
        'signature_separator': re.compile(r'^-- $', re.MULTILINE),
        'newsgroups_header': re.compile(r'^Newsgroups:.*$', re.MULTILINE),
        'subject_header': re.compile(r'^Subject:.*$', re.MULTILINE),
        'from_header': re.compile(r'^From:.*$', re.MULTILINE),
        'organization_header': re.compile(r'^Organization:.*$', re.MULTILINE),
        'lines_header': re.compile(r'^Lines:.*$', re.MULTILINE),
        'reply_header': re.compile(r'^In article <.*>.*$', re.MULTILINE),
        'wrote_line': re.compile(r'^\s*\w+ writes?:', re.MULTILINE),
        'citation': re.compile(r'^".*"$', re.MULTILINE),
        'code_block': re.compile(r'^[ \t]+[\w\W]+?$', re.MULTILINE),
        'technical_term': re.compile(r'\b(?:PCI|ISA|VLB|SCSI|IDE|ATAPI|BIOS|CMOS|RAM|CPU|MHz|GB|MB|KB)\b', re.IGNORECASE),
        'file_extension': re.compile(r'\.\w{2,4}\b'),
        'version_number': re.compile(r'\b\d+\.\d+(?:\.\d+)?\b'),
        'hex_number': re.compile(r'\b0x[0-9a-fA-F]+\b'),
        'binary_number': re.compile(r'\b0b[01]+\b'),
    }
    results = {}
    special_chars_counter = Counter()
    for text in sample_texts:
        for name, pattern in patterns.items():
            if name not in results:
                results[name] = []
            matches = pattern.findall(text)
            if matches:
                results[name].extend(matches)
        special = re.findall(r'[^\w\s]', text)
        special_chars_counter.update(special)
    return results, special_chars_counter

results, special_counter = analyze(df['text'].tolist())
for char, count in special_counter.most_common(20):
    print(f"  '{char if char != '\n' else '\\n'}': {count} раз")

  '-': 15148 раз
  '.': 13958 раз
  ',': 5579 раз
  ':': 5093 раз
  '=': 3941 раз
  '>': 2950 раз
  ')': 2798 раз
  '(': 2602 раз
  '*': 2598 раз
  '@': 2032 раз
  '/': 1759 раз
  ''': 1592 раз
  '|': 1228 раз
  '"': 1049 раз
  '?': 952 раз
  '!': 818 раз
  ';': 494 раз
  '+': 483 раз
  '~': 397 раз
  '<': 345 раз


In [87]:
word_freq = find_repeat(df['text'].tolist())

for word, count in word_freq.most_common(30):
    print(f"  '{word}': {count}")

  'the': 19105
  'to': 10748
  'a': 10000
  'of': 8515
  'and': 8386
  'I': 7567
  'is': 6693
  'in': 5056
  'for': 5046
  'X': 4675
  'that': 4038
  'on': 3616
  '>': 3585
  'it': 3401
  'with': 3384
  'you': 3019
  'have': 2853
  'be': 2826
  '|': 2619
  'Subject:': 2521
  'are': 2476
  'The': 2439
  'or': 2421
  'From:': 2414
  'this': 2388
  'Lines:': 2346
  'Organization:': 2287
  '-': 2123
  'not': 2024
  'can': 1990


In [88]:
test_texts = df['text'].iloc[:5].tolist()
for i, text in enumerate(test_texts):
    print(f"\nОригинал {i+1}: {text[:100]}...")
    cleaned = clean(text)
    print(f"Очищенный {i+1}: {cleaned[:100]}...")


Оригинал 1: From: bgrubb@dante.nmsu.edu (GRUBB)
Subject: Re: IDE vs SCSI
Organization: New Mexico State Universi...
Очищенный 1: bgrubb nmsu edu grubb subject ide vs scsi organization new mexico state university las cruces nm lin...

Оригинал 2: From: DonH@cup.portal.com (Don - Hirschfeld)
Subject: Re: Toshiba 3401B CD-ROM:  Any problems?
Organ...
Очищенный 2: donh portal com hirschfeld subject toshiba 3401b cd rom problems organization portal system tm lines...

Оригинал 3: From: christy@cs.concordia.ca (Christy)
Subject: XFree86 --- need help...
Organization: Computer Sci...
Очищенный 3: christy concordia ca christy subject xfree86 need help organization computer science concordia unive...

Оригинал 4: From: arp@cooper!osd (Andrew Pinkowitz)
Subject: SIGGRAPH -- Conference on Understanding Images
Keyw...
Очищенный 4: arp osd andrew pinkowitz subject siggraph conference understanding images keywords graphics animatio...

Оригинал 5: From: Peter.vanderveen@visser.el.wau.nl  (Peter van

4 Задание - избавиться от частей речи

In [89]:
test_text = df['text'].iloc[0][:500]
print(f"Исходный текст:\n{test_text}\n")
nouns_adjs = filter_by_pos(test_text, ['N', 'J'])
nouns_adjs_verbs = filter_by_pos(test_text, ['N', 'J', 'V'])
print(nouns_adjs)
print(nouns_adjs_verbs)

Исходный текст:
From: bgrubb@dante.nmsu.edu (GRUBB)
Subject: Re: IDE vs SCSI
Organization: New Mexico State University, Las Cruces, NM
Lines: 49
Distribution: world
NNTP-Posting-Host: dante.nmsu.edu

DXB132@psuvm.psu.edu writes:
>SCSI-I ranges from 0-5MB/s.
>SCSI-II ranges from 0-40MB/s.
>IDE ranges from 0-8.3MB/s.                                       
>ESDI is always 1.25MB/s (although there are some non-standard versions)
The above does not tell the proper story of SCSI:
SCSI-I: 8-bit asynchronous {~1.5MB/s 

bgrubb grubb subject re ide vs scsi organization new mexico state university las cruces nm lines distribution world ranges ranges ide ranges esdi versions above proper story scsi asynchronous
bgrubb grubb subject re ide vs scsi organization new mexico state university las cruces nm lines distribution world writes ranges ranges ide ranges esdi is are versions above does tell proper story scsi asynchronous


In [72]:
df['nouns_adjs'] = df['text'].apply(lambda x: filter_by_pos(x, ['N', 'J']))
df['nouns_adjs_verbs'] = df['text'].apply(lambda x: filter_by_pos(x, ['N', 'J', 'V']))
for i in range(3):
    print(f"\nДокумент {i+1}")
    print(f"Оригинал: {df['text'].iloc[i][:200]}")
    print(f"Сущ+прил: {df['nouns_adjs'].iloc[i][:200]}")
    print(f"Сущ+прил+гл: {df['nouns_adjs_verbs'].iloc[i][:200]}")


Документ 1
Оригинал: From: bgrubb@dante.nmsu.edu (GRUBB)
Subject: Re: IDE vs SCSI
Organization: New Mexico State University, Las Cruces, NM
Lines: 49
Distribution: world
NNTP-Posting-Host: dante.nmsu.edu

DXB132@psuvm.psu
Сущ+прил: bgrubb grubb subject re ide vs scsi organization new mexico state university las cruces nm lines distribution world ranges ranges ide ranges esdi versions above proper story scsi asynchronous synchron
Сущ+прил+гл: bgrubb grubb subject re ide vs scsi organization new mexico state university las cruces nm lines distribution world writes ranges ranges ide ranges esdi is are versions above does tell proper story sc

Документ 2
Оригинал: From: DonH@cup.portal.com (Don - Hirschfeld)
Subject: Re: Toshiba 3401B CD-ROM:  Any problems?
Organization: The Portal System (TM)
Lines: 1

I have the PAS16 / Toshiba 3401 combo and have no problems
Сущ+прил: donh don hirschfeld subject re toshiba problems organization portal system tm lines toshiba combo problems
Сущ+прил+гл: